# 03 - Uji akhir pada data TEST (dijalankan sekali)
Yang diuji adalah **bundel export** `bundle/fraud_xgb_mask_v1/` (juara resmi, lihat `02c`), bukan objek di notebook. Dengan begitu yang dinilai adalah artefak yang benar-benar akan di-deploy.

**Aturan main**
* Data `data/test/` (seed 8, 20.000 nasabah baru, ID `T...`) belum pernah dipakai untuk tuning, seleksi resep, maupun penentuan ambang. Integritasnya diverifikasi dengan SHA-256 saat dimuat.
* Tidak ada penyetelan apa pun setelah melihat hasil ini. Bila model gagal, hasil dilaporkan apa adanya.
* Model lain (challenger HGB, XGB tanpa augmentasi, LR, rule gate §6.3) ikut dilaporkan **hanya sebagai pembanding**, tidak untuk memilih ulang juara.

**Kriteria lulus, ditetapkan sebelum data TEST dimuat** (angka acuan dari DEV-OOT: bersih 0,654, 20% hilang 0,506):
| # | Kriteria | Ambang |
|---|---|---|
| C1 | PR-AUC bersih | >= 0,50 dan batas bawah CI95 >= 0,40 |
| C2 | PR-AUC saat 20% nilai hilang | >= 0,35 |
| C3 | Lebih baik dari rule gate §6.3 | batas bawah CI95 model > batas atas CI95 rule gate |
| C4 | Presisi pita `high_risk` | >= 0,30 (ambang dari DEV-OOT menargetkan 0,50) |
| C5 | Fairness (kesamaan peluang): rasio recall terbesar/terkecil antar kelompok yang punya >= 30 hari fraud | <= 1,25 antar gender dan <= 1,5 antar kelompok usia |

*Koreksi C5 sebelum TEST dibuka:* versi awal menuntut **laju alert** sama antar kelompok. Latihan seluruh notebook ini pada data DEV-OOT (bukan TEST) menunjukkan kriteria itu keliru secara konsep, karena prevalensi fraud memang berbeda antar usia (data dibangun dengan kecenderungan ini), sehingga laju alert wajar berbeda. Yang adil dinilai adalah **recall** (peluang fraud tertangkap) per kelompok. Laju alert tetap dilaporkan sebagai informasi.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")      # notebooks use paths relative to the project root
print("working directory:", os.getcwd())

In [1]:
import os, sys, json, warnings, joblib
os.environ.setdefault("OMP_NUM_THREADS", "8"); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd()); import splits as S
from sklearn.metrics import roc_auc_score, average_precision_score

BUNDLE, ART, L, W, ALERT, N_BOOT, SEED = "bundle/fraud_xgb_mask_v1", "artifacts", S.LABEL, S.WEIGHT, 0.001, 200, 42
CRIT = dict(C1_pr=0.50, C1_ci_low=0.40, C2_miss20=0.35, C4_high_precision=0.30, C5_gender=1.25, C5_age=1.5, C5_min_fraud_days=30)       # tetap, jangan diubah setelah melihat hasil
sys.path.insert(0, BUNDLE); from score import FraudScorer
sc = FraudScorer(BUNDLE); FEATS = sc.features; CATS = sc.cats
NUM = [f for f in FEATS if f not in CATS]
print("juara:", sc.schema["model"], "| fitur:", len(FEATS), "| ambang:", sc.schema["thresholds"])

juara: XGB|mask | fitur: 68 | ambang: {'suspicious': 0.002535842126235366, 'high_risk': 0.05366142466664314, 'note': "berlaku pada 'score' (terkoreksi prior); suspicious = presisi >= 5% di DEV-OOT, high_risk = presisi >= 50%"}


In [2]:
test = S.load_test(i_understand_this_is_the_final_evaluation=True)          # memverifikasi hash MANIFEST sebelum membaca
man = json.load(open(f"{S.TEST}/MANIFEST.json"))
print(f"TEST: {len(test):,} baris | nasabah {test.customer_id.nunique():,} | nasabah fraud (hari fraud) {test.loc[test[L] == 1, 'customer_id'].nunique()} | baris fraud {int(test[L].sum()):,}")
print(f"prevalensi asli {np.average(test[L], weights=test[W]):.4%} | hash diverifikasi: {list(man['sha256'])}")
y, w = test[L].values, test[W].values

TEST: 195,698 baris | nasabah 19,495 | nasabah fraud (hari fraud) 585 | baris fraud 2,050
prevalensi asli 0.0317% | hash diverifikasi: ['daydetect.parquet', 'snapshots.parquet', 'customers.csv', 'fraud_events.csv']


In [3]:
class Model:      # kerangka kosong agar pickle pembanding dapat dibuka
    pass
def as_cats(d):   # samakan kategori dengan skema training
    d = d.copy()
    for c, cats in CATS.items(): d[c] = pd.Categorical(d[c].astype(object).where(d[c].astype(object).isin(cats)), categories=cats)
    return d
def joblib_score(path, d):
    m = joblib.load(path); d = as_cats(d)
    return m.m.predict_proba(d[NUM] if m.name == "LR" else d[FEATS])[:, 1]
def rule_gate(d):
    h = ((d.new_device_7 > 0).astype(int) + (d.cred_change_3 > 0) + (d.ftp_large_7 > 0) + (d.night_share_7 > 0.3)
         + (d.geo_mismatch_share_7 > 0.5) + (d.out_to_balance_max_3d > 0.5) + (d.new_payee_cnt_7 >= 2))
    return h.values + 0.01 * d.out_to_balance_1.values
def metrics(y, s, w, alert=ALERT):
    o = np.argsort(-s); ys, ws = y[o], w[o]; k = np.searchsorted(np.cumsum(ws), alert * ws.sum()); tp = (ys[:k] * ws[:k]).sum()
    return dict(PR_AUC=average_precision_score(y, s, sample_weight=w), ROC_AUC=roc_auc_score(y, s, sample_weight=w), recall_at_alert=tp / (ys * ws).sum(), precision_at_alert=tp / ws[:k].sum())

X = test[FEATS]
res_c = sc.score_df(X, reasons=False)                                                    # juara: lewat bundel
scores = {"JUARA XGB mask (bundel)": res_c.score_raw.values,
          "XGB tanpa augmentasi": joblib_score("models/baseline_XGB_clean.joblib", X),
          "HGB mask+grup+noise (challenger)": joblib_score("models/baseline_HGB_mask+grup+noise.joblib", X),
          "Logistic Regression": joblib_score("models/baseline_LR.joblib", X),
          "Rule gate 6.3": rule_gate(test)}
tab = pd.DataFrame({k: metrics(y, s, w) for k, s in scores.items()}).T; tab

,PR_AUC,ROC_AUC,recall_at_alert,precision_at_alert
JUARA XGB mask (bundel),0.716223,0.998228,0.851707,0.270866
XGB tanpa augmentasi,0.726783,0.998398,0.857561,0.273633
HGB mask+grup+noise (challenger),0.713559,0.998559,0.863902,0.275100
Logistic Regression,0.318840,0.954774,0.447805,0.142296
Rule gate 6.3,0.015178,0.856852,0.058537,0.018595


In [4]:
# CI95 bootstrap per nasabah (200 resampel), dan selisih berpasangan terhadap juara
codes, n_c = pd.factorize(test.customer_id)[0], test.customer_id.nunique(); rng = np.random.default_rng(SEED); B = {k: [] for k in scores}
for _ in range(N_BOOT):
    wb = w * np.bincount(rng.integers(0, n_c, n_c), minlength=n_c)[codes]
    for k, s in scores.items(): B[k].append(average_precision_score(y, s, sample_weight=wb))
B = pd.DataFrame(B); champ = "JUARA XGB mask (bundel)"
ci = pd.DataFrame({"PR_AUC": tab.PR_AUC, "CI95_low": B.quantile(.025), "CI95_high": B.quantile(.975)}).round(4); display(ci)
print("Selisih PR-AUC juara minus pembanding [2.5%, median, 97.5%]:")
display(pd.DataFrame({k: (B[champ] - B[k]).quantile([.025, .5, .975]).round(4).values for k in scores if k != champ}, index=["2.5%", "median", "97.5%"]))

,PR_AUC,CI95_low,CI95_high
JUARA XGB mask (bundel),0.7162,0.6393,0.7807
XGB tanpa augmentasi,0.7268,0.6652,0.7906
HGB mask+grup+noise (challenger),0.7136,0.6468,0.7797
Logistic Regression,0.3188,0.2636,0.3903
Rule gate 6.3,0.0152,0.0102,0.0217


Selisih PR-AUC juara minus pembanding [2.5%, median, 97.5%]:


,XGB tanpa augmentasi,HGB mask+grup+noise (challenger),Logistic Regression,Rule gate 6.3
2.5%,-0.0374,-0.0231,0.3502,0.6256
median,-0.0089,0.0017,0.3973,0.7119
97.5%,0.0184,0.0298,0.4443,0.7656


### Pita risiko dengan ambang beku dari DEV-OOT

In [5]:
b = res_c.band.values; d = pd.DataFrame({"band": b, "w": w, "fraud": w * y}).groupby("band").sum().reindex(["genuine", "suspicious", "high_risk"]).fillna(0)
d["share_of_traffic"] = d.w / d.w.sum(); d["precision"] = d.fraud / d.w.replace(0, np.nan); d["share_of_fraud_caught"] = d.fraud / d.fraud.sum()
display(d[["share_of_traffic", "precision", "share_of_fraud_caught"]].style.format("{:.3%}"))
pos = test[y == 1].assign(band=b[y == 1])
print("Tangkapan fraud per tipe dan varian stealth (baris = hari fraud; kolom = pita yang menerima):")
display(pd.crosstab([pos._fraud_type, pos._stealth], pos.band, normalize="index").reindex(columns=["genuine", "suspicious", "high_risk"]).fillna(0).round(3))

,share_of_traffic,precision,share_of_fraud_caught
band,,,
genuine,99.573%,0.002%,6.976%
suspicious,0.385%,1.428%,17.317%
high_risk,0.042%,57.087%,75.707%


Tangkapan fraud per tipe dan varian stealth (baris = hari fraud; kolom = pita yang menerima):


band                  genuine  suspicious  high_risk
_fraud_type _stealth                                
ato         0           0.133       0.143      0.724
            1           0.270       0.532      0.198
mule        0           0.036       0.078      0.886
            1           0.103       0.209      0.688
scam        0           0.018       0.122      0.860
            1           0.065       0.265      0.670

### Ketahanan terhadap data tidak lengkap (skenario sama dengan 02b)

In [6]:
def mask_cells(d, rate, rng):
    d = d.copy()
    for c in FEATS: d[c] = d[c].mask(rng.random(len(d)) < rate)
    return d
def blank(d, cols):
    d = d.copy()
    for c in cols: d[c] = d[c].mask(np.ones(len(d), bool))
    return d
def noise(d, sg, rng):
    d = d.copy(); d[NUM] = d[NUM].astype("float64") * np.exp(rng.normal(0, sg, (len(d), len(NUM)))); return d
g = sc.groups; r = np.random.default_rng(400); Xc = as_cats(X)
SC = {"clean": Xc, "miss20": mask_cells(Xc, .2, r), "tanpa_payee": blank(Xc, g["payee"]), "noise15": noise(Xc, .15, r), "miss20+noise15": noise(mask_cells(Xc, .2, r), .15, r)}
clean_model = joblib.load("models/baseline_XGB_clean.joblib")
rows = {}
for sname, dd in SC.items():
    s_champ = sc.score_df(dd, reasons=False).score_raw.values; s_base = clean_model.m.predict_proba(dd[FEATS])[:, 1]
    rows[sname] = dict(juara_PR=metrics(y, s_champ, w)["PR_AUC"], tanpa_augmentasi_PR=metrics(y, s_base, w)["PR_AUC"], juara_recall_alert=metrics(y, s_champ, w)["recall_at_alert"])
    if sname == "miss20": s_miss = s_champ
rob = pd.DataFrame(rows).T.round(4); display(rob)
wb_ci = []
for _ in range(N_BOOT):
    wb = w * np.bincount(rng.integers(0, n_c, n_c), minlength=n_c)[codes]; wb_ci.append(average_precision_score(y, s_miss, sample_weight=wb))
miss_ci = np.quantile(wb_ci, [.025, .975]).round(4); print("PR-AUC juara saat 20% hilang, CI95:", miss_ci)

,juara_PR,tanpa_augmentasi_PR,juara_recall_alert
clean,0.7162,0.7268,0.8517
miss20,0.5349,0.1989,0.7132
tanpa_payee,0.4589,0.3769,0.6361
noise15,0.4591,0.4176,0.6185
miss20+noise15,0.3226,0.0647,0.4971


PR-AUC juara saat 20% hilang, CI95: [0.4571 0.6236]


### Fairness (recall dan laju alert per kelompok, pada ambang alert 0,1% traffic)

In [7]:
s = scores[champ]; thr = np.sort(s)[::-1][np.searchsorted(np.cumsum(w[np.argsort(-s)]), ALERT * w.sum())]; flag = s >= thr
fair = []
for col in ("gender", "age_ord"):
    for v, gdf in test.assign(f=flag).groupby(col):
        gp = gdf[gdf[L] == 1]; fair.append(dict(group=f"{col}={v}", flag_rate=np.average(gdf.f, weights=gdf[W]), recall=gp.f.mean() if len(gp) else np.nan, n_fraud_days=len(gp)))
fair = pd.DataFrame(fair).set_index("group"); display(fair.round(4))
def ratio(p, col="recall"):
    g_ = fair[fair.index.str.startswith(p) & (fair.n_fraud_days >= CRIT["C5_min_fraud_days"])][col]; return g_.max() / g_.min()
print(f"rasio recall terbesar/terkecil (C5): gender {ratio('gender'):.2f} | usia {ratio('age_ord'):.2f}    [info] rasio laju alert: gender {ratio('gender', 'flag_rate'):.2f} | usia {ratio('age_ord', 'flag_rate'):.2f}")

,flag_rate,recall,n_fraud_days
group,,,
gender=F,0.0012,0.8645,1173
gender=M,0.0008,0.8347,877
age_ord=0,0.0017,0.8781,517
age_ord=1,0.0009,0.8417,581
age_ord=2,0.0009,0.8104,422
age_ord=3,0.0007,0.8635,271
age_ord=4,0.0012,0.8764,259


rasio recall terbesar/terkecil (C5): gender 1.04 | usia 1.08    [info] rasio laju alert: gender 1.36 | usia 2.51


## Keputusan lulus-gagal (kriteria ditetapkan di awal notebook)

In [8]:
c_pr, c_lo = ci.loc[champ, "PR_AUC"], ci.loc[champ, "CI95_low"]
checks = {
  "C1 PR-AUC bersih >= 0,50 dan CI bawah >= 0,40": (c_pr >= CRIT["C1_pr"] and c_lo >= CRIT["C1_ci_low"], f"{c_pr:.3f} (CI {c_lo:.3f}-{ci.loc[champ, 'CI95_high']:.3f})"),
  "C2 PR-AUC 20% hilang >= 0,35": (rob.loc["miss20", "juara_PR"] >= CRIT["C2_miss20"], f"{rob.loc['miss20', 'juara_PR']:.3f} (CI {miss_ci[0]:.3f}-{miss_ci[1]:.3f})"),
  "C3 CI bawah juara > CI atas rule gate": (ci.loc[champ, "CI95_low"] > ci.loc["Rule gate 6.3", "CI95_high"], f"{ci.loc[champ, 'CI95_low']:.3f} vs {ci.loc['Rule gate 6.3', 'CI95_high']:.3f}"),
  "C4 presisi high_risk >= 0,30": (d.loc["high_risk", "precision"] >= CRIT["C4_high_precision"] if d.loc["high_risk", "w"] > 0 else False, f"{d.loc['high_risk', 'precision']:.3f}"),
  "C5 fairness recall: gender <= 1,25 dan usia <= 1,5": (ratio("gender") <= CRIT["C5_gender"] and ratio("age_ord") <= CRIT["C5_age"], f"gender {ratio('gender'):.2f}, usia {ratio('age_ord'):.2f}"),
}
verdict = pd.DataFrame({k: dict(lulus="LULUS" if ok else "GAGAL", nilai=v) for k, (ok, v) in checks.items()}).T; display(verdict)
print("HASIL AKHIR:", "SEMUA KRITERIA LULUS" if all(ok for ok, _ in checks.values()) else "ADA KRITERIA GAGAL - lihat tabel")

,lulus,nilai
"C1 PR-AUC bersih >= 0,50 dan CI bawah >= 0,40",LULUS,0.716 (CI 0.639-0.781)
"C2 PR-AUC 20% hilang >= 0,35",LULUS,0.535 (CI 0.457-0.624)
C3 CI bawah juara > CI atas rule gate,LULUS,0.639 vs 0.022
"C4 presisi high_risk >= 0,30",LULUS,0.571
"C5 fairness recall: gender <= 1,25 dan usia <= 1,5",LULUS,"gender 1.04, usia 1.08"


HASIL AKHIR: SEMUA KRITERIA LULUS


In [9]:
out = dict(test_manifest_sha=man["sha256"], criteria=CRIT, pr_auc=ci.to_dict(), table=tab.round(4).to_dict(), bands=d[["share_of_traffic", "precision", "share_of_fraud_caught"]].round(5).to_dict(),
           robustness=rob.to_dict(), fairness=fair.round(5).to_dict(), verdict={k: dict(ok=bool(ok), value=v) for k, (ok, v) in checks.items()})
json.dump(out, open(f"{ART}/test_results.json", "w"), indent=1, default=str)
card = open(f"{BUNDLE}/MODEL_CARD.md").read().split("## Hasil TEST")[0]
card += "## Hasil TEST (data baru, seed 8; dijalankan sekali)\n" + f"* PR-AUC bersih {c_pr:.3f} (CI95 {c_lo:.3f}-{ci.loc[champ, 'CI95_high']:.3f}); 20% nilai hilang {rob.loc['miss20', 'juara_PR']:.3f}; tanpa payee {rob.loc['tanpa_payee', 'juara_PR']:.3f}; rule gate §6.3 {ci.loc['Rule gate 6.3', 'PR_AUC']:.3f}.\n"
card += "* Kriteria: " + "; ".join(f"{k.split(' ')[0]} {'lulus' if ok else 'GAGAL'}" for k, (ok, _) in checks.items()) + ".\n* Detail: `artifacts/test_results.json`, notebook `03_final_test.ipynb`.\n"
open(f"{BUNDLE}/MODEL_CARD.md", "w").write(card); print("tersimpan: artifacts/test_results.json dan MODEL_CARD.md diperbarui")

tersimpan: artifacts/test_results.json dan MODEL_CARD.md diperbarui
